# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [ ]:
import sys
import os

# Ensure src/ is on the path so `lasr` is importable.
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from lasr.config import ModelConfig, InferenceConfig, PromptStyle, SAEConfig
from lasr.data import load_esnli, build_few_shot_examples, build_prompts
from lasr.inference import load_model, generate_predictions
from lasr.metrics import run_evaluation
from lasr.sae import load_sae
from lasr.activations import gather_residual_activations, encode_activations
from lasr.aggregation import top_k_features, reconstruction_metrics, l0_sparsity
from lasr.plotting import plot_feature_activation_heatmap

# Configuration

In [ ]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT
use_few_shot = True

sae_config = SAEConfig(
    layer=40,
    width="65k",
    l0="medium",
    repo_id="google/gemma-scope-2-27b-it",
)

ESNLI_URL = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/refs/heads/master/dataset/esnli_dev.csv"

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

# Setup — HF Token

In [ ]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [ ]:
esnli_df = load_esnli(ESNLI_URL)
esnli_df.head()

# Build Prompts

In [ ]:
few_shot_examples = build_few_shot_examples(esnli_df, prompt_style) if use_few_shot else None
esnli_df["prompt"] = build_prompts(esnli_df, prompt_style, few_shot=use_few_shot, few_shot_examples=few_shot_examples)

print(esnli_df["prompt"].iloc[0])

# Load Model

In [ ]:
model, tokenizer = load_model(model_config)

# Inference — Chain of Thought (Few-shot)

In [ ]:
valid_mask = esnli_df["prompt"].notna()
valid_indices = esnli_df.index[valid_mask][:: inference_config.downsample_rate]
sampled_mask = esnli_df.index.isin(valid_indices)

prompts = esnli_df.loc[sampled_mask, "prompt"].tolist()
print(f"Running on {len(prompts)} / {valid_mask.sum()} samples (1 in every {inference_config.downsample_rate})")

decoded_outputs = generate_predictions(
    prompts, model, tokenizer, inference_config, device=model_config.device
)

esnli_df, report = run_evaluation(
    esnli_df, decoded_outputs, sampled_mask, prompt_style,
    model_name=model_config.model_name, few_shot=use_few_shot,
)
print(report)

# Inference — One-Word (Zero-shot)

In [ ]:
ow_prompt_style = PromptStyle.ONE_WORD
ow_few_shot = False

esnli_df["prompt"] = build_prompts(esnli_df, ow_prompt_style, few_shot=ow_few_shot)

ow_prompts = esnli_df.loc[sampled_mask, "prompt"].tolist()
print(f"Running on {len(ow_prompts)} / {valid_mask.sum()} samples (1 in every {inference_config.downsample_rate})")

ow_decoded_outputs = generate_predictions(
    ow_prompts, model, tokenizer, inference_config, device=model_config.device
)

esnli_df, ow_report = run_evaluation(
    esnli_df, ow_decoded_outputs, sampled_mask, ow_prompt_style,
    model_name=model_config.model_name, few_shot=ow_few_shot,
)
print(ow_report)

# SAE Activation Analysis

In [ ]:
sae = load_sae(sae_config)

## Gather activations for generated tokens

In [ ]:
import torch

# Pick the first sample prompt
sample_prompt = esnli_df["prompt"].iloc[0]
prompt_ids = tokenizer.encode(sample_prompt, return_tensors="pt", add_special_tokens=True).to(model_config.device)
prompt_len = prompt_ids.shape[1]
print(f"Prompt tokens: {prompt_len}")

# Generate the model's response (returns prompt + generated tokens)
full_ids = model.generate(input_ids=prompt_ids, max_new_tokens=inference_config.max_new_tokens)
gen_len = full_ids.shape[1] - prompt_len
print(f"Generated tokens: {gen_len}")
print(f"Full sequence tokens: {full_ids.shape[1]}")

# Run a forward pass on the full sequence and hook the SAE target layer
residual_acts = gather_residual_activations(model, sae_config.layer, full_ids)
print(f"Residual activations shape (full): {residual_acts.shape}")

# Slice to keep only the generated-token activations
gen_acts = residual_acts[:, prompt_len:, :]
print(f"Generated-only activations shape: {gen_acts.shape}")

# Encode through the SAE
sae_acts, reconstruction = encode_activations(sae, gen_acts)
print(f"SAE feature activations shape: {sae_acts.shape}")
print(f"Reconstruction shape: {reconstruction.shape}")

In [ ]:
# Reconstruction quality
metrics = reconstruction_metrics(reconstruction, gen_acts)
print(f"MSE: {metrics['mse']:.6f}")
print(f"FVU: {metrics['fvu']:.6f}")

In [ ]:
# L0 sparsity
l0 = l0_sparsity(sae_acts)
print(f"Per-token L0: {l0}")
print(f"Average L0: {l0.float().mean():.1f}")

In [ ]:
# Top-K features (averaged across generated tokens)
TOP_K = 10
top_vals, top_indices = top_k_features(sae_acts, k=TOP_K)
print(f"Top {TOP_K} SAE features (by mean activation across generated tokens):")
for rank, (idx, val) in enumerate(zip(top_indices.tolist(), top_vals.tolist()), 1):
    print(f"  #{rank}  feature {idx:>5d}  activation = {val:.4f}")

## Activation Heatmap

In [ ]:
# Decode generated token strings
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)

# Build activation matrix: (k, gen_len) for the top features
feature_indices = top_indices.tolist()
act_matrix = sae_acts[0, :, top_indices].T  # (k, gen_len)

fig = plot_feature_activation_heatmap(
    act_matrix,
    tokens=tokens,
    feature_indices=feature_indices,
    title="Gemma3-27b-it SAE Feature Activations (Generated Tokens, Layer 40)",
)
fig.show()